[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abpandey-dev/data-engineering-from-scratch-to-advanced/blob/main/notebooks/topics/11_spark-foundations.ipynb)

*Data Engineering from Scratch to Advanced · Notebook 11 of 21*

# Spark Foundations

Apache Spark from zero: the driver and executors, lazy evaluation, transformations vs actions, narrow vs wide, DataFrames and PySpark's API. You build a tiny Spark in plain Python to see how it works, then run the real thing in the labs.

### 👩‍🏫 From your teacher

**The picture to keep in your head:** Spark writes a plan first and works later. Data moving across the network is what costs you.

**What interviewers really test here:** Lazy evaluation, narrow vs wide, jobs, stages and tasks, and how many partitions to use.

### How to use this notebook

1. **Run the setup cell** just below.
2. **Read each lesson slowly.** Work the numbers on paper before reading on.
3. **Run and then change the examples.** Breaking code on purpose is how you learn.
4. **Try each practice question before opening an answer.** If you're stuck, open one hint at a time.
5. **Say your interview answers out loud.** Knowing something and explaining it are different skills.

### Your roadmap

- [ ] 1. What Spark is, and how a job runs *(Basics)*
- [ ] 2. Lazy evaluation: Spark plans before it works *(Core)*
- [ ] 3. Narrow and wide: the idea behind Spark performance *(Core)*
- [ ] 4. The DataFrame API you'll use daily *(Core)*
- [ ] 5. Partitions and sizing a cluster *(Advanced)*
- [ ] 20 practice questions
- [ ] Interview questions, out loud

In [1]:
# ▶ Run this cell first. It prepares the notebook; you don't need to read it.
import contextlib, copy, io, math, sys
import numpy as np, matplotlib.pyplot as plt
%matplotlib inline
import math
import numpy as np
import matplotlib.pyplot as plt
PLAIN = True

class ListNode:
    def __init__(self, val=0, next=None):
        self.val, self.next = val, next

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def mk_list(vals, pos=-1):
    dummy = cur = ListNode()
    nodes = []
    for v in vals:
        cur.next = ListNode(v)
        cur = cur.next
        nodes.append(cur)
    if pos >= 0 and nodes:
        nodes[-1].next = nodes[pos]
    return dummy.next

def list_arr(head):
    out, seen = [], set()
    while head and id(head) not in seen:
        seen.add(id(head))
        out.append(head.val)
        head = head.next
    return out

def mk_tree(vals):
    if not vals or vals[0] is None:
        return None
    root = TreeNode(vals[0])
    queue, i = [root], 1
    while queue and i < len(vals):
        node = queue.pop(0)
        for side in ("left", "right"):
            if i < len(vals) and vals[i] is not None:
                child = TreeNode(vals[i])
                setattr(node, side, child)
                queue.append(child)
            i += 1
    return root

def tree_arr(root):
    out, queue = [], [root]
    while queue:
        node = queue.pop(0)
        if node:
            out.append(node.val)
            queue += [node.left, node.right]
        else:
            out.append(None)
    while out and out[-1] is None:
        out.pop()
    return out

def find(root, val):
    if not root or root.val == val:
        return root
    return find(root.left, val) or find(root.right, val)

ListNode.__repr__ = lambda self: f"ListNode{list_arr(self)}"
TreeNode.__repr__ = lambda self: f"TreeNode{tree_arr(self)}"

def plain(x, digits=4):
    """NumPy / pandas results as plain Python values, so they compare with a test's expected answer."""
    np = sys.modules.get("numpy")
    pd = sys.modules.get("pandas")
    if pd is not None:
        if isinstance(x, pd.DataFrame):
            return {str(c): plain(x[c].tolist(), digits) for c in x.columns}
        if isinstance(x, pd.Series):
            return {plain(k, digits): plain(v, digits) for k, v in x.items()}
        if isinstance(x, pd.Index):
            return plain(x.tolist(), digits)
        if x is pd.NaT:
            return None
        if isinstance(x, pd.Timestamp):
            return str(x.date()) if x == x.normalize() else str(x)
    if np is not None:
        if isinstance(x, np.ndarray):
            return plain(x.tolist(), digits)
        if isinstance(x, np.generic):
            return plain(x.item(), digits)
    if isinstance(x, float):
        return None if x != x else round(x, digits) + 0.0
    if isinstance(x, list):
        return [plain(v, digits) for v in x]
    if isinstance(x, tuple):
        return tuple(plain(v, digits) for v in x)
    if isinstance(x, dict):
        return {plain(k, digits): plain(v, digits) for k, v in x.items()}
    if isinstance(x, (set, frozenset)):
        return {plain(v, digits) for v in x}
    return x


def check(fn, tests, plain_compare=None, listify=False, design=False, prep=None, post=None,
          norm=None, verify=None, runner=None, inplace=None):
    """Run your code against the tests and show which pass."""
    use_plain = PLAIN if plain_compare is None else plain_compare
    passed, problems = 0, []
    for args, want in tests:
        try:
            a = copy.deepcopy(args)
            if design:
                obj = fn(*a[0])
                got = [getattr(obj, m)(*p) for m, p in a[1]]
            elif runner is not None:
                got = runner(fn, a)
            else:
                if prep is not None:
                    a = prep(a)
                with contextlib.redirect_stdout(io.StringIO()):
                    got = fn(*a)
                if inplace is not None:
                    got = a[inplace]
                if listify:
                    got = list(got)
                if post is not None:
                    got = post(got)
            if verify is not None:
                ok = bool(verify(copy.deepcopy(args), got))
            elif norm is not None:
                ok = norm(copy.deepcopy(got)) == norm(copy.deepcopy(want))
            else:
                ok = (plain(got), plain(want)) if use_plain else (got, want)
                ok = ok[0] == ok[1]
            ok = ok if isinstance(ok, bool) else False
        except Exception as e:  # your code raised an error: report it, don't stop the notebook
            ok, got = False, f"{type(e).__name__}: {e}"
        passed += ok
        if not ok:
            problems.append((args, want, got))
    plt.close("all")
    if passed == len(tests):
        print(f"✅ All {len(tests)} tests passed. Well done! Now compare with the answers below.")
    else:
        print(f"❌ {passed}/{len(tests)} tests passed. First one to fix:")
        args, want, got = problems[0]
        print("   input:   ", repr(args)[:300])
        print("   expected:", repr(want)[:300])
        print("   you got: ", repr(got)[:300])
    return None

print("Ready! Scroll down and start learning.")

Ready! Scroll down and start learning.


---
# Part 1: The theory in one place

Read this once for the big picture. It's fine if it doesn't all stick yet: the lessons will make it concrete, and you can come back to revise.

### RDD, DataFrame, Dataset

| API | What | Use |
| --- | --- | --- |
| RDD | low-level collection of objects | rarely; no query optimisation |
| DataFrame | table with a schema, optimised | almost always |
| Dataset | typed DataFrame (Scala/Java) | when type safety matters |

---
# Part 2: Lessons, step by step

## Lesson 1: What Spark is, and how a job runs

*Level: Basics*

Spark lets you write what looks like ordinary data code, and it runs across hundreds of machines. It's the most widely used big data engine, so it's a core skill.



```text
        ┌ Driver: your program. Makes the plan.
        │
Cluster │  Executor   Executor   Executor
        └  [task][task] [task][task] [task][task]
            partition  partition  partition
```



| Part | Role |
| --- | --- |
| Driver | runs your code, plans the work, gathers results |
| Executor | a process on a worker machine that runs tasks |
| Task | one unit of work on one partition, using one core |
| Partition | a chunk of data |



**The chain:** an action starts a **job**, which is split into **stages** at each shuffle, and each stage has one **task** per partition.



10 tasks of 60 s on 4 cores take 3 rounds: **180 s**.



**Your first PySpark, as used in the lab:**



```python
from pyspark.sql import SparkSession, functions as F

spark = SparkSession.builder.appName("orders").getOrCreate()
df = spark.read.parquet("s3://lake/orders/")
out = (df.filter(F.col("amount") > 0)
         .groupBy("city")
         .agg(F.sum("amount").alias("total")))
out.write.mode("overwrite").parquet("s3://lake/city_totals/")
```



This course's browser IDE can't run Spark, so questions here use a miniature Spark in Python, which shows the mechanics directly. **Run the real thing with `labs/spark_lab.py`**, or in Colab.

**✏️ Your turn:** you now have what you need for [Q1](#q1), [Q2](#q2), [Q5](#q5). Try one now, or carry on and do them in Part 3.

## Lesson 2: Lazy evaluation: Spark plans before it works

*Level: Core*

**Transformations** (filter, select, join, groupBy) only describe work. **Actions** (count, collect, write, show) make it happen. In the chain filter, select, count, groupBy, write: **2 jobs** run.



In our mini-Spark, mapping and filtering 1, 2, 3 does **0** work until collect, and then 6 operations.



**Why wait? Because seeing the whole plan lets Spark optimise it.** The Catalyst optimiser rewrites your query:



| Optimisation | What it does | Example saving |
| --- | --- | --- |
| **Predicate pushdown** | moves filters toward the data | expensive step handles 3 rows rather than 6 |
| **Column pruning** | reads only needed columns | 16 KB rather than 546 KB |
| **Partition pruning** | skips folders | read 1 day rather than 365 |



**Read the plan with `df.explain()`:** learn to look for these:



| In the plan | Meaning |
| --- | --- |
| `Exchange` | a shuffle: expensive |
| `PushedFilters: [...]` | the file reader's filtering early |
| `PartitionFilters` | folders are being skipped |
| `BroadcastHashJoin` | no shuffle for that join |
| `SortMergeJoin` | both sides were shuffled and sorted |



**Trap:** applying a Python UDF, or casting a column in a filter, can stop pushdown working.

**✏️ Your turn:** you now have what you need for [Q8](#q8), [Q9](#q9), [Q10](#q10), [Q3](#q3), [Q20](#q20). Try one now, or carry on and do them in Part 3.

## Lesson 3: Narrow and wide: the idea behind Spark performance

*Level: Core*

**If you remember one thing about Spark performance, remember this.**



|   | Narrow | Wide (shuffle) |
| --- | --- | --- |
| Data moves? | no | **yes, across the network** |
| Examples | filter, select, withColumn, map | groupBy, join, distinct, sort, repartition |
| Cost | cheap | expensive: network, disk, memory |



Each wide operation starts a new stage. read, filter, groupBy, select, join, write has 2 wide steps, so it's **3 stages**.



**A shuffle, step by step:** to group by key, all rows with the same key must reach the same machine. Each task splits its output by `hash(key) % partitions` and writes it to disk, and the next stage's tasks then fetch their share.



**groupBy is smarter than it looks.** Each partition first sums locally, so it sends far less. Rows a:1, a:2, b:3 and a:4 send only 3 partial results rather than 4 raw rows. With millions of rows per key, this saving is huge. `collect_list` can't benefit, which is one reason it's expensive.



**Fault tolerance.** Spark remembers each partition's **lineage**: how it was built. If an executor dies, only the lost partitions are recomputed, by rerunning their steps. No replication of the working data is needed.

**✏️ Your turn:** you now have what you need for [Q4](#q4), [Q11](#q11), [Q12](#q12), [Q13](#q13), [Q19](#q19). Try one now, or carry on and do them in Part 3.

## Lesson 4: The DataFrame API you'll use daily

*Level: Core*

```python
from pyspark.sql import functions as F, Window

df = spark.read.option("header", True).csv(path)      # or .parquet, .json
df.printSchema()

clean = (df
    .withColumn("amount", F.col("amount").cast("double"))
    .filter(F.col("amount").isNotNull())
    .withColumn("day", F.to_date("ts"))
    .dropDuplicates(["id"]))

totals = clean.groupBy("city", "day").agg(
    F.sum("amount").alias("total"),
    F.countDistinct("user").alias("users"))

w = Window.partitionBy("city").orderBy(F.desc("day"))
latest = totals.withColumn("rn", F.row_number().over(w)).filter("rn = 1")

(latest.write.mode("overwrite")
    .partitionBy("day").parquet(out_path))
```



**You can also just use SQL:** `df.createOrReplaceTempView("t"); spark.sql("SELECT ...")`. It compiles to the same plan.



**Arrays:** `F.explode` gives one row per item, dropping empty ones (use `explode_outer` to keep them).



**Avoid Python UDFs where you can.** Each row has to travel from the JVM to Python and back. With built-ins at 200 ms and UDFs at 5,000 ms per million rows, it's **25 times slower** here. Check `pyspark.sql.functions` first. If you must, a pandas UDF works in batches, and is much faster.



**Never `collect()` big data.** It pulls everything to the driver, which crashes. Use `limit` or write it out.



**Cache only when it pays:** reused 5 times, a 100 s build costs 500 s without a cache and 120 s with one. Used once, caching makes it slower. Remember `unpersist`.

**✏️ Your turn:** you now have what you need for [Q14](#q14), [Q7](#q7), [Q6](#q6), [Q17](#q17). Try one now, or carry on and do them in Part 3.

## Lesson 5: Partitions and sizing a cluster

*Level: Advanced*

**Too few partitions:** tasks are huge, and may run out of memory. **Too many:** scheduling overhead and tiny files. A common target is around **128 MB per partition**, so 100 GB needs about 782.



**Rule of thumb:** at least 2–4 times as many partitions as total cores, so that tasks balance out. With 10 executors of 4 cores, 40 tasks run at once.



|   | coalesce(n) | repartition(n) |
| --- | --- | --- |
| Shuffle? | **no** | yes |
| Can grow? | no, only shrink | yes |
| Result | may be uneven | even |
| Sizes 10, 10, 100, 10 to 2 | 20, 110 | 65, 65 |



Use coalesce to cheaply reduce file count, and repartition to fix skew, or to control the layout by a column.



**Sizing executors.** On nodes of 16 cores and 64 GB, keep 1 core and 1 GB for the system. A common starting point is about 5 cores per executor, as too many cores per executor hurts I/O. That's 3 executors per node, of 21 GB each, less about 10% overhead: **18.9 GB**.



```python
spark-submit --num-executors 30 --executor-cores 5 \
  --executor-memory 18g --driver-memory 4g job.py
```



These are starting points. Measure with the Spark UI, and adjust.

**✏️ Your turn:** you now have what you need for [Q15](#q15), [Q16](#q16), [Q18](#q18). Try one now, or carry on and do them in Part 3.

---
# Part 3: Practice like an interview

There are 20 questions here, easiest first.

**Work the way you would in a real interview:** restate the problem, try an example by hand, say the simple approach and its cost, improve it, then write the code. Test it with the cell. Only open an answer once you've made a real attempt, because struggling is where the learning happens.

<a id="q1"></a>
### Q1. Split Data Into Partitions

🟢 Warm-up · *The unit of parallelism*

Split a list into n partitions in turn: item i goes to partition i % n.

**Examples**

- `rows = [1, 2, 3, 4, 5], n = 2` → `[[1, 3, 5], [2, 4]]`

In [ ]:
def split_partitions(rows, n):
    """
    Split a list into n partitions in turn: item i goes to partition i % n.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(split_partitions, [
    (([1, 2, 3, 4, 5], 2), [[1, 3, 5], [2, 4]]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A list of lists.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Slicing with a step.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** A list of lists.

```python
def split_partitions(rows, n):
    parts = [[] for _ in range(n)]
    for i, r in enumerate(rows):
        parts[i % n].append(r)
    return parts
```

**Cost:** O(k) time, O(k) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Slicing with a step.

```python
def split_partitions(rows, n):
    return [rows[i::n] for i in range(n)]
```

**Cost:** O(k) time, O(k) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A list of lists. That costs O(k) time and O(k) space."

"I can do better: Slicing with a step. That's O(k) time and O(k) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/spark-foundations/split-data-into-partitions)

<a id="q2"></a>
### Q2. Map Over Partitions

🟢 Warm-up · *Narrow transformation*

Double every value in each partition, keeping partitions as they are.

**Examples**

- `parts = [[1, 2], [3]]` → `[[2, 4], [6]]`

In [ ]:
def map_partitions(parts):
    """
    Double every value in each partition, keeping partitions as they are.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(map_partitions, [
    (([[1, 2], [3]],), [[2, 4], [6]]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Nested loops.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A nested comprehension. No data moves between partitions.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Nested loops.

```python
def map_partitions(parts):
    out = []
    for p in parts:
        new = []
        for x in p:
            new.append(x * 2)
        out.append(new)
    return out
```

**Cost:** O(k) time, O(k) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A nested comprehension. No data moves between partitions.

```python
def map_partitions(parts):
    return [[x * 2 for x in p] for p in parts]
```

**Cost:** O(k) time, O(k) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Nested loops. That costs O(k) time and O(k) space."

"I can do better: A nested comprehension. No data moves between partitions. That's O(k) time and O(k) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/spark-foundations/map-over-partitions)

<a id="q3"></a>
### Q3. Column Pruning

🟢 Warm-up · *Read only what you need*

Given columns needed downstream and each column's bytes per row, return bytes read for n rows (needed only), and bytes saved compared with all columns.

**Examples**

- `sizes = {'id': 8, 'name': 30, 'bio': 500, 'amt': 8}, needed = ['id', 'amt'], n = 1000` → `(16000, 530000)`

In [ ]:
def column_prune(sizes, needed, n):
    """
    Given columns needed downstream and each column's bytes per row, return
    bytes read for n rows (needed only), and bytes saved compared with all
    columns.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(column_prune, [
    (({'id': 8, 'name': 30, 'bio': 500, 'amt': 8}, ['id', 'amt'], 1000), (16000, 530000)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A loop.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Generators.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** A loop.

```python
def column_prune(sizes, needed, n):
    used = 0
    for c in needed:
        used += sizes[c]
    total = 0
    for c in sizes:
        total += sizes[c]
    return (used * n, (total - used) * n)
```

**Cost:** O(c) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Generators.

```python
def column_prune(sizes, needed, n):
    used = sum(sizes[c] for c in needed)
    return (used * n, (sum(sizes.values()) - used) * n)
```

**Cost:** O(c) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A loop. That costs O(c) time and O(1) space."

"I can do better: Generators. That's O(c) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/spark-foundations/column-pruning)

<a id="q4"></a>
### Q4. Narrow or Wide?

🟢 Warm-up · *Does data move?*

Classify operations: filter, map, select, withColumn and union are 'narrow'; groupBy, join, distinct, sort and repartition are 'wide'.

**Examples**

- `ops = ['filter', 'groupBy', 'select', 'join']` → `['narrow', 'wide', 'narrow', 'wide']`

In [ ]:
def classify_ops(ops):
    """
    Classify operations: filter, map, select, withColumn and union are 'narrow';
    groupBy, join, distinct, sort and repartition are 'wide'.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(classify_ops, [
    ((['filter', 'groupBy', 'select', 'join'],), ['narrow', 'wide', 'narrow', 'wide']),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Ifs.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A set.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Ifs.

```python
def classify_ops(ops):
    out = []
    for op in ops:
        if op in ("filter", "map", "select", "withColumn", "union"):
            out.append("narrow")
        else:
            out.append("wide")
    return out
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A set.

```python
def classify_ops(ops):
    wide = {"groupBy", "join", "distinct", "sort", "repartition"}
    return ["wide" if o in wide else "narrow" for o in ops]
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Ifs. That costs O(n) time and O(n) space."

"I can do better: A set. That's O(n) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/spark-foundations/narrow-or-wide)

<a id="q5"></a>
### Q5. Parallel Time on a Cluster

🟢 Warm-up · *Tasks and slots*

n equal tasks, each t seconds, run on c cores in rounds. Return total seconds.

**Examples**

- `n = 10, t = 60, c = 4` → `180`
- `n = 4, t = 60, c = 4` → `60`

In [ ]:
def cluster_time(n, t, c):
    """
    n equal tasks, each t seconds, run on c cores in rounds. Return total
    seconds.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(cluster_time, [
    ((10, 60, 4), 180),
    ((4, 60, 4), 60),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Simulate rounds.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`ceil(n / c) × t`.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Simulate rounds.

```python
def cluster_time(n, t, c):
    remaining, total = n, 0
    while remaining > 0:
        remaining -= c
        total += t
    return total
```

**Cost:** O(n/c) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `ceil(n / c) × t`.

```python
def cluster_time(n, t, c):
    return math.ceil(n / c) * t
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Simulate rounds. That costs O(n/c) time and O(1) space."

"I can do better: `ceil(n / c) × t`. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/spark-foundations/parallel-time-on-a-cluster)

<a id="q6"></a>
### Q6. UDF vs Built-in Cost

🟢 Warm-up · *Why avoid Python UDFs*

A Python UDF serialises each row out of the JVM and back. Given rows, built-in ms per million, UDF ms per million, return both times in seconds, and the ratio rounded to 1 decimal.

**Examples**

- `rows = 10000000, builtin_ms = 200, udf_ms = 5000` → `(2.0, 50.0, 25.0)`

In [ ]:
def udf_cost(rows, builtin_ms, udf_ms):
    """
    A Python UDF serialises each row out of the JVM and back. Given rows, built-
    in ms per million, UDF ms per million, return both times in seconds, and the
    ratio rounded to 1 decimal.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(udf_cost, [
    ((10000000, 200, 5000), (2.0, 50.0, 25.0)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Step by step.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Compact.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Step by step.

```python
def udf_cost(rows, builtin_ms, udf_ms):
    millions = rows / 1000000
    a = millions * builtin_ms / 1000
    b = millions * udf_ms / 1000
    return (a, b, round(b / a, 1))
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Compact.

```python
def udf_cost(rows, builtin_ms, udf_ms):
    m = rows / 1e9
    return (m * builtin_ms, m * udf_ms, round(udf_ms / builtin_ms, 1))
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Step by step. That costs O(1) time and O(1) space."

"I can do better: Compact. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/spark-foundations/udf-vs-built-in-cost)

<a id="q7"></a>
### Q7. Explode and Collect

🟢 Warm-up · *Array columns*

Implement explode (one row per array item, dropping empty arrays, as Spark's explode does) on (id, list) rows, then collect\_list back by id.

**Examples**

- `rows = [(1, ['a', 'b']), (2, []), (3, ['c'])]` → `([[1, 'a'], [1, 'b'], [3, 'c']], {1: ['a', 'b'], 3: ['c']})`

In [ ]:
def explode_collect(rows):
    """
    Implement explode (one row per array item, dropping empty arrays, as Spark's
    explode does) on (id, list) rows, then collect_list back by id.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(explode_collect, [
    (([(1, ['a', 'b']), (2, []), (3, ['c'])],), ([[1, 'a'], [1, 'b'], [3, 'c']], {1: ['a', 'b'], 3: ['c']})),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Nested loops.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Comprehension, plus a defaultdict.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Nested loops.

```python
def explode_collect(rows):
    exploded = []
    for i, items in rows:
        for x in items:
            exploded.append([i, x])
    back = {}
    for i, x in exploded:
        back.setdefault(i, []).append(x)
    return (exploded, back)
```

**Cost:** O(t) time, O(t) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Comprehension, plus a defaultdict.

```python
def explode_collect(rows):
    from collections import defaultdict
    exploded = [[i, x] for i, items in rows for x in items]
    back = defaultdict(list)
    for i, x in exploded:
        back[i].append(x)
    return (exploded, dict(back))
```

**Cost:** O(t) time, O(t) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Nested loops. That costs O(t) time and O(t) space."

"I can do better: Comprehension, plus a defaultdict. That's O(t) time and O(t) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/spark-foundations/explode-and-collect)

<a id="q8"></a>
### Q8. Action or Transformation?

🟢 Warm-up · *What triggers a job?*

count, collect, show, write and first are actions, and they start a job. Given a chain, return how many jobs run.

**Examples**

- `chain = ['filter', 'select', 'count', 'groupBy', 'write']` → `2`
- `chain = ['filter', 'select']` → `0`

In [ ]:
def jobs_run(chain):
    """
    count, collect, show, write and first are actions, and they start a job.
    Given a chain, return how many jobs run.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(jobs_run, [
    ((['filter', 'select', 'count', 'groupBy', 'write'],), 2),
    ((['filter', 'select'],), 0),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A loop with a tuple check.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A set and a sum.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** A loop with a tuple check.

```python
def jobs_run(chain):
    jobs = 0
    for op in chain:
        if op in ("count", "collect", "show", "write", "first"):
            jobs += 1
    return jobs
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A set and a sum.

```python
def jobs_run(chain):
    return sum(op in {"count", "collect", "show", "write", "first"} for op in chain)
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A loop with a tuple check. That costs O(n) time and O(1) space."

"I can do better: A set and a sum. That's O(n) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/spark-foundations/action-or-transformation)

<a id="q9"></a>
### Q9. Lazy Evaluation

🟡 Interview standard · *Nothing runs until an action*

Build a tiny lazy dataset: map and filter only record the plan, and collect runs it. Return (steps run before collect, result, values processed in total) for mapping ×2 and filtering &gt; 2 on the given rows.

**Examples**

- `rows = [1, 2, 3]` → `(0, [4, 6], 6)`

In [ ]:
def lazy_demo(rows):
    """
    Build a tiny lazy dataset: map and filter only record the plan, and collect
    runs it. Return (steps run before collect, result, values processed in
    total) for mapping ×2 and filtering > 2 on the given rows.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(lazy_demo, [
    (([1, 2, 3],), (0, [4, 6], 6)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A class holding a list of operations.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Generators are lazy by nature, doing the same in fewer lines.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** A class holding a list of operations.

```python
def lazy_demo(rows):
    class Lazy:
        def __init__(self, data):
            self.data, self.plan, self.ran = data, [], 0

        def map(self, f):
            self.plan.append(("map", f))
            return self

        def filter(self, f):
            self.plan.append(("filter", f))
            return self

        def collect(self):
            out = []
            for x in self.data:
                keep = True
                for kind, f in self.plan:
                    self.ran += 1
                    if kind == "map":
                        x = f(x)
                    elif not f(x):
                        keep = False
                        break
                if keep:
                    out.append(x)
            return out
    d = Lazy(rows).map(lambda x: x * 2).filter(lambda x: x > 2)
    before = d.ran
    result = d.collect()
    return (before, result, d.ran)
```

**Cost:** O(n·s) time, O(s) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Generators are lazy by nature, doing the same in fewer lines.

```python
def lazy_demo(rows):
    count = [0]

    def double(x):
        count[0] += 1
        return x * 2

    def keep(x):
        count[0] += 1
        return x > 2
    plan = filter(keep, map(double, rows))
    before = count[0]
    result = list(plan)
    return (before, result, count[0])
```

**Cost:** O(n·s) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A class holding a list of operations. That costs O(n·s) time and O(s) space."

"I can do better: Generators are lazy by nature, doing the same in fewer lines. That's O(n·s) time and O(1) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `rows = [1, 2, 3]`

1. double() returns 2
2. keep() returns False
3. double() returns 4
4. keep() returns True
5. double() returns 6
6. keep() returns True
7. lazy\_demo() returns (0, [4, 6], 6)

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/spark-foundations/lazy-evaluation)

<a id="q10"></a>
### Q10. Predicate Pushdown

🟡 Interview standard · *Why laziness saves work*

Spark can move a filter before an expensive step. Return records processed by the expensive step, with the filter after it, and before it.

**Examples**

- `rows = [1, 2, 3, 4, 5, 6], modulus = 2` → `(6, 3)`

In [ ]:
def pushdown(rows, modulus):
    """
    Spark can move a filter before an expensive step. Return records processed
    by the expensive step, with the filter after it, and before it.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(pushdown, [
    (([1, 2, 3, 4, 5, 6], 2), (6, 3)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Run both orders, with counters.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

It's simply n and the number passing the filter.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Run both orders, with counters.

```python
def pushdown(rows, modulus):
    after = 0
    for r in rows:
        after += 1
    before = 0
    for r in rows:
        if r % modulus == 0:
            before += 1
    return (after, before)
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** It's simply n and the number passing the filter.

```python
def pushdown(rows, modulus):
    return (len(rows), sum(r % modulus == 0 for r in rows))
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Run both orders, with counters. That costs O(n) time and O(n) space."

"I can do better: It's simply n and the number passing the filter. That's O(n) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/spark-foundations/predicate-pushdown)

<a id="q11"></a>
### Q11. Stages From a Plan

🟡 Interview standard · *Shuffles cut stages*

A job's stages are separated by wide operations. Return the number of stages for a list of operations.

**Examples**

- `ops = ['read', 'filter', 'groupBy', 'select', 'join', 'write']` → `3`
- `ops = ['read', 'filter', 'write']` → `1`

In [ ]:
def count_stages(ops):
    """
    A job's stages are separated by wide operations. Return the number of stages
    for a list of operations.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(count_stages, [
    ((['read', 'filter', 'groupBy', 'select', 'join', 'write'],), 3),
    ((['read', 'filter', 'write'],), 1),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Count with a loop.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A generator.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Count with a loop.

```python
def count_stages(ops):
    stages = 1
    for op in ops:
        if op in ("groupBy", "join", "distinct", "sort", "repartition"):
            stages += 1
    return stages
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A generator.

```python
def count_stages(ops):
    wide = {"groupBy", "join", "distinct", "sort", "repartition"}
    return 1 + sum(o in wide for o in ops)
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Count with a loop. That costs O(n) time and O(1) space."

"I can do better: A generator. That's O(n) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/spark-foundations/stages-from-a-plan)

<a id="q12"></a>
### Q12. Shuffle by Key

🟡 Interview standard · *Bring keys together*

Move (key, value) pairs across partitions so that each key lands in partition (sum of char codes) % n. Return partitions, and how many records changed partition.

**Examples**

- `parts = [[('a', 1), ('b', 2)], [('a', 3)]], n = 2` → `([[('b', 2)], [('a', 1), ('a', 3)]], 1)`

In [ ]:
def shuffle(parts, n):
    """
    Move (key, value) pairs across partitions so that each key lands in
    partition (sum of char codes) % n. Return partitions, and how many records
    changed partition.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(shuffle, [
    (([[('a', 1), ('b', 2)], [('a', 3)]], 2), ([[('b', 2)], [('a', 1), ('a', 3)]], 1)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Loops.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A helper for the target partition.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Loops.

```python
def shuffle(parts, n):
    out = [[] for _ in range(n)]
    moved = 0
    for i, p in enumerate(parts):
        for k, v in p:
            target = sum(ord(c) for c in k) % n
            out[target].append((k, v))
            if target != i:
                moved += 1
    return (out, moved)
```

**Cost:** O(k) time, O(k) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A helper for the target partition.

```python
def shuffle(parts, n):
    target = lambda k: sum(map(ord, k)) % n
    out = [[] for _ in range(n)]
    moved = 0
    for i, p in enumerate(parts):
        for kv in p:
            t = target(kv[0])
            out[t].append(kv)
            moved += t != i
    return (out, moved)
```

**Cost:** O(k) time, O(k) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Loops. That costs O(k) time and O(k) space."

"I can do better: A helper for the target partition. That's O(k) time and O(k) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/spark-foundations/shuffle-by-key)

<a id="q13"></a>
### Q13. groupBy + sum, Spark Style

🟡 Interview standard · *Partial, shuffle, final*

Each partition first sums per key locally, then those partial results are merged. Return (partial records sent, final dictionary sorted). Partial aggregation is why groupBy shuffles less than the raw data.

**Examples**

- `parts = [[('a', 1), ('a', 2), ('b', 3)], [('a', 4)]]` → `(3, {'a': 7, 'b': 3})`

In [ ]:
def group_sum(parts):
    """
    Each partition first sums per key locally, then those partial results are
    merged. Return (partial records sent, final dictionary sorted). Partial
    aggregation is why groupBy shuffles less than the raw data.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(group_sum, [
    (([[('a', 1), ('a', 2), ('b', 3)], [('a', 4)]],), (3, {'a': 7, 'b': 3})),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Merge everything directly, sending all records.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Local dictionaries, then merge.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Merge everything directly, sending all records.

```python
def group_sum(parts):
    sent = 0
    final = {}
    for p in parts:
        local = {}
        for k, v in p:
            local[k] = local.get(k, 0) + v
        sent += len(local)
        for k, v in local.items():
            final[k] = final.get(k, 0) + v
    return (sent, dict(sorted(final.items())))
```

**Cost:** O(k) time, O(g) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Local dictionaries, then merge.

```python
def group_sum(parts):
    from collections import Counter
    locals_ = []
    for p in parts:
        c = Counter()
        for k, v in p:
            c[k] += v
        locals_.append(c)
    return (sum(map(len, locals_)), dict(sorted(sum(locals_, Counter()).items())))
```

**Cost:** O(k) time, O(g) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Merge everything directly, sending all records. That costs O(k) time and O(g) space."

"I can do better: Local dictionaries, then merge. That's O(k) time and O(g) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/spark-foundations/groupby-sum-spark-style)

<a id="q14"></a>
### Q14. A Tiny DataFrame

🟡 Interview standard · *select, filter, withColumn*

Implement filter(amt &gt; min), add column tax = amt × 0.1, then select id and tax, on a list of dictionaries. Round tax to 2 decimals.

**Examples**

- `rows = [{'id': 1, 'amt': 100, 'x': 'a'}, {'id': 2, 'amt': 5, 'x': 'b'}], minimum = 10` → `[{'id': 1, 'tax': 10.0}]`

In [ ]:
def tiny_dataframe(rows, minimum):
    """
    Implement filter(amt > min), add column tax = amt × 0.1, then select id and
    tax, on a list of dictionaries. Round tax to 2 decimals.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(tiny_dataframe, [
    (([{'id': 1, 'amt': 100, 'x': 'a'}, {'id': 2, 'amt': 5, 'x': 'b'}], 10), [{'id': 1, 'tax': 10.0}]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

One loop with all steps.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Chained comprehensions, as in a DataFrame pipeline.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** One loop with all steps.

```python
def tiny_dataframe(rows, minimum):
    out = []
    for r in rows:
        if r["amt"] > minimum:
            out.append({"id": r["id"], "tax": round(r["amt"] * 0.1, 2)})
    return out
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Chained comprehensions, as in a DataFrame pipeline.

```python
def tiny_dataframe(rows, minimum):
    filtered = (r for r in rows if r["amt"] > minimum)
    with_tax = ({**r, "tax": round(r["amt"] * 0.1, 2)} for r in filtered)
    return [{k: r[k] for k in ("id", "tax")} for r in with_tax]
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: One loop with all steps. That costs O(n) time and O(n) space."

"I can do better: Chained comprehensions, as in a DataFrame pipeline. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/spark-foundations/a-tiny-dataframe)

<a id="q15"></a>
### Q15. How Many Partitions?

🟡 Interview standard · *A common rule of thumb*

Target about 128 MB per partition. Return partitions for data\_gb, and how many tasks run at a time with executors × cores. GB = 1000 MB.

**Examples**

- `data_gb = 100, executors = 10, cores = 4` → `(782, 40)`

In [ ]:
def partition_plan(data_gb, executors, cores):
    """
    Target about 128 MB per partition. Return partitions for data_gb, and how
    many tasks run at a time with executors × cores. GB = 1000 MB.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(partition_plan, [
    ((100, 10, 4), (782, 40)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Step by step.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Compact.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Step by step.

```python
def partition_plan(data_gb, executors, cores):
    mb = data_gb * 1000
    parts = mb // 128
    if mb % 128:
        parts += 1
    return (parts, executors * cores)
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Compact.

```python
def partition_plan(data_gb, executors, cores):
    return (math.ceil(data_gb * 1000 / 128), executors * cores)
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Step by step. That costs O(1) time and O(1) space."

"I can do better: Compact. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/spark-foundations/how-many-partitions)

<a id="q16"></a>
### Q16. Sizing an Executor

🟡 Interview standard · *Memory per core*

Reserving 1 core and 1 GB per node for the system, with executors of 5 cores each: return (executors per node, memory per executor rounded down, minus 10% overhead, rounded to 1 decimal).

**Examples**

- `node_cores = 16, node_gb = 64` → `(3, 18.9)`

In [ ]:
def size_executor(node_cores, node_gb):
    """
    Reserving 1 core and 1 GB per node for the system, with executors of 5 cores
    each: return (executors per node, memory per executor rounded down, minus
    10% overhead, rounded to 1 decimal).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(size_executor, [
    ((16, 64), (3, 18.9)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Divide step by step.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Compact.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Divide step by step.

```python
def size_executor(node_cores, node_gb):
    cores = node_cores - 1
    gb = node_gb - 1
    executors = cores // 5
    each = gb // executors
    return (executors, round(each * 0.9, 1))
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Compact.

```python
def size_executor(node_cores, node_gb):
    n = (node_cores - 1) // 5
    return (n, round((node_gb - 1) // n * 0.9, 1))
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Divide step by step. That costs O(1) time and O(1) space."

"I can do better: Compact. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/spark-foundations/sizing-an-executor)

<a id="q17"></a>
### Q17. Cache or Recompute?

🟡 Interview standard · *Spend memory to save time*

A dataset costs build seconds to compute, and is used n times. Caching adds cache\_cost once, with each use then taking read seconds. Return (seconds without, seconds with, whether to cache).

**Examples**

- `build = 100, n = 5, cache_cost = 10, read = 2` → `(500, 120, True)`
- `build = 100, n = 1, cache_cost = 10, read = 2` → `(100, 112, False)`

In [ ]:
def cache_or_not(build, n, cache_cost, read):
    """
    A dataset costs build seconds to compute, and is used n times. Caching adds
    cache_cost once, with each use then taking read seconds. Return (seconds
    without, seconds with, whether to cache).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(cache_or_not, [
    ((100, 5, 10, 2), (500, 120, True)),
    ((100, 1, 10, 2), (100, 112, False)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Compute both.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Compact.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Compute both.

```python
def cache_or_not(build, n, cache_cost, read):
    without = build * n
    with_cache = build + cache_cost + read * n
    return (without, with_cache, with_cache < without)
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Compact.

```python
def cache_or_not(build, n, cache_cost, read):
    a, b = build * n, build + cache_cost + read * n
    return (a, b, b < a)
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Compute both. That costs O(1) time and O(1) space."

"I can do better: Compact. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/spark-foundations/cache-or-recompute)

<a id="q18"></a>
### Q18. Coalesce vs Repartition

🟡 Interview standard · *Shrinking without a shuffle*

coalesce merges neighbouring partitions without a shuffle; repartition shuffles evenly. Given sizes and target n, return the coalesce result (groups of neighbours, in order, as evenly as possible by count) and the repartition result (even split of the total, remainder to the first).

**Examples**

- `sizes = [10, 10, 100, 10], n = 2` → `([20, 110], [65, 65])`

In [ ]:
def coalesce_vs_repartition(sizes, n):
    """
    coalesce merges neighbouring partitions without a shuffle; repartition
    shuffles evenly. Given sizes and target n, return the coalesce result
    (groups of neighbours, in order, as evenly as possible by count) and the
    repartition result (even split of the total, remainder to the first).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(coalesce_vs_repartition, [
    (([10, 10, 100, 10], 2), ([20, 110], [65, 65])),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Loops.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Slices and divmod.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Loops.

```python
def coalesce_vs_repartition(sizes, n):
    coalesced = []
    size = len(sizes) // n
    extra = len(sizes) % n
    start = 0
    for i in range(n):
        end = start + size + (1 if i < extra else 0)
        coalesced.append(sum(sizes[start:end]))
        start = end
    total = sum(sizes)
    repart = [total // n + (1 if i < total % n else 0) for i in range(n)]
    return (coalesced, repart)
```

**Cost:** O(p) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Slices and divmod.

```python
def coalesce_vs_repartition(sizes, n):
    q, r = divmod(len(sizes), n)
    bounds = [i * q + min(i, r) for i in range(n + 1)]
    Q_, R = divmod(sum(sizes), n)
    return ([sum(sizes[a:b]) for a, b in zip(bounds, bounds[1:])], [Q_ + (i < R) for i in range(n)])
```

**Cost:** O(p) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Loops. That costs O(p) time and O(n) space."

"I can do better: Slices and divmod. That's O(p) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/spark-foundations/coalesce-vs-repartition)

<a id="q19"></a>
### Q19. Lineage: Recompute a Lost Partition

🟡 Interview standard · *Fault tolerance*

ops is a list of steps applied to an input. If an executor dies, Spark reapplies them to the lost partition's input. Return (recovered values, recompute count).

**Examples**

- `inputs = [1, 2], ops = ['double', 'inc']` → `([3, 5], 4)`

In [ ]:
def recompute(inputs, ops):
    """
    ops is a list of steps applied to an input. If an executor dies, Spark
    reapplies them to the lost partition's input. Return (recovered values,
    recompute count).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(recompute, [
    (([1, 2], ['double', 'inc']), ([3, 5], 4)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Apply with ifs.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A dictionary of functions.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Apply with ifs.

```python
def recompute(inputs, ops):
    out, count = [], 0
    for x in inputs:
        for op in ops:
            count += 1
            if op == "double":
                x = x * 2
            else:
                x = x + 1
        out.append(x)
    return (out, count)
```

**Cost:** O(n·s) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A dictionary of functions.

```python
def recompute(inputs, ops):
    funcs = {"double": lambda x: x * 2, "inc": lambda x: x + 1}
    out = []
    for x in inputs:
        for op in ops:
            x = funcs[op](x)
        out.append(x)
    return (out, len(inputs) * len(ops))
```

**Cost:** O(n·s) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Apply with ifs. That costs O(n·s) time and O(n) space."

"I can do better: A dictionary of functions. That's O(n·s) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/spark-foundations/lineage-recompute-a-lost-partition)

<a id="q20"></a>
### Q20. Read a Plan

🟡 Interview standard · *Does it use pushdown?*

Given lines of a physical plan, return (number of Exchange (shuffle) steps, whether PushedFilters is non-empty).

**Examples**

- `lines = ['*(2) HashAggregate', '+- Exchange hashpartitioning(city)', '   +- FileScan parquet PushedFilters: [IsNotNull(amt)]']` → `(1, True)`
- `lines = ['FileScan parquet PushedFilters: []']` → `(0, False)`

In [ ]:
def read_plan(lines):
    """
    Given lines of a physical plan, return (number of Exchange (shuffle) steps,
    whether PushedFilters is non-empty).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(read_plan, [
    ((['*(2) HashAggregate', '+- Exchange hashpartitioning(city)', '   +- FileScan parquet PushedFilters: [IsNotNull(amt)]'],), (1, True)),
    ((['FileScan parquet PushedFilters: []'],), (0, False)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Search each line.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A regular expression.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Search each line.

```python
def read_plan(lines):
    exchanges, pushed = 0, False
    for line in lines:
        if "Exchange" in line:
            exchanges += 1
        if "PushedFilters: [" in line and "PushedFilters: []" not in line:
            pushed = True
    return (exchanges, pushed)
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A regular expression.

```python
def read_plan(lines):
    import re
    text = "\n".join(lines)
    return (text.count("Exchange"), bool(re.search(r"PushedFilters: \[[^\]]+\]", text)))
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Search each line. That costs O(n) time and O(1) space."

"I can do better: A regular expression. That's O(n) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/spark-foundations/read-a-plan)

---
# Part 4: Interview questions, out loud

Cover the answer, **say yours out loud**, then open it and compare. Aim for two or three clear sentences, then an example.

<details>
<summary><b>1. Transformations vs actions?</b></summary>

Transformations are lazy and build a plan. Actions trigger execution and return or write results.

</details>

<details>
<summary><b>2. Why is Spark lazy?</b></summary>

Seeing the whole plan allows optimisations, such as pushing filters down and pruning columns, before any work is done.

</details>

<details>
<summary><b>3. Narrow vs wide transformation?</b></summary>

Narrow needs no data movement. Wide needs a shuffle, which starts a new stage and costs network, disk and memory.

</details>

<details>
<summary><b>4. Job, stage, task?</b></summary>

An action creates a job. Shuffles split it into stages. Each stage has one task per partition.

</details>

<details>
<summary><b>5. What does the driver do?</b></summary>

It runs your program, builds the plan, schedules tasks and collects results. It's a single point of failure and should not hold large data.

</details>

<details>
<summary><b>6. How does Spark handle a failed executor?</b></summary>

It reruns lost tasks using lineage on other executors, retrying up to a limit.

</details>

<details>
<summary><b>7. coalesce vs repartition?</b></summary>

coalesce reduces partitions without a shuffle, and may be uneven. repartition does a full shuffle, and can grow or shrink, giving an even result.

</details>

<details>
<summary><b>8. Why avoid Python UDFs?</b></summary>

Rows are serialised between the JVM and Python, which is slow and blocks some optimisations. Use built-in functions, or pandas UDFs if necessary.

</details>

<details>
<summary><b>9. cache vs persist?</b></summary>

cache stores at the default level. persist lets you choose memory, disk or both. Both are lazy, and only worthwhile if data is reused.

</details>

<details>
<summary><b>10. Scenario: your job OOMs on the driver. What's the likely cause?</b></summary>

A collect, toPandas or big broadcast pulling too much data to the driver. Write results out instead, or limit them.

</details>

<details>
<summary><b>11. Scenario: a count takes as long as the whole job. Why?</b></summary>

Because of laziness, it runs all preceding transformations. Avoid unnecessary counts, or cache if you truly need the data several times.

</details>


---
# Check yourself before moving on

You're ready for the next notebook when you can tick these **without looking**:

- [ ] I can explain: What Spark is, and how a job runs
- [ ] I can explain: Lazy evaluation: Spark plans before it works
- [ ] I can explain: Narrow and wide: the idea behind Spark performance
- [ ] I can explain: The DataFrame API you'll use daily
- [ ] I can explain: Partitions and sizing a cluster

**Revision drill.** Come back in a few days, set a 25-minute timer per question, and solve these from a blank cell:

- [ ] [Q16](#q16) Sizing an Executor
- [ ] [Q17](#q17) Cache or Recompute?
- [ ] [Q18](#q18) Coalesce vs Repartition
- [ ] [Q19](#q19) Lineage: Recompute a Lost Partition
- [ ] [Q20](#q20) Read a Plan

Spaced repetition beats re-reading: redoing a question after you've partly forgotten it is what makes it stick.